# PS-2 GPU embedding extraction for MOT17 (Colab, Tesla T4)

Embeds every **public detection** (SDP detector `det.txt`) of the MOT17 train sequences with the PS-1 OSNet embedder on
the GPU. Embeddings are cached (`emb`, `emb_flip`, rows aligned with `det.txt`) so the tracker + TrackEval run on CPU
locally. Run order: this notebook's cells top to bottom on a T4 runtime (CPU fallback works, much slower).
The code-transfer instructions are identical to `ps1_colab.ipynb` (Git clone of a pinned commit, or zip upload).

In [ ]:
# 1. Verify the GPU
import subprocess, sys
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "no nvidia-smi: CPU runtime")
import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU - falling back to CPU (slow). Switch runtime to T4 for the intended run.")

In [ ]:
# 2. Config
SOURCE = "git"          # "git" or "zip"
REPO_URL = "https://github.com/<you>/<repo>.git"   # <-- set (git mode)
COMMIT = "main"         # <-- pin to an exact commit SHA for reproducibility
MODELS = [("x1_0", "osnet_x1_0_msmt17"), ("x0_25", "osnet_x0_25_msmt17"), ("x1_0", "osnet_x1_0_imagenet")]
HF_W = "https://huggingface.co/kaiyangzhou/osnet/resolve/main/"
WEIGHT_FILES = {
    "osnet_x1_0_msmt17": "osnet_x1_0_msmt17_combineall_256x128_amsgrad_ep150_stp60_lr0.0015_b64_fb10_softmax_labelsmooth_flip_jitter.pth",
    "osnet_x0_25_msmt17": "osnet_x0_25_msmt17_combineall_256x128_amsgrad_ep150_stp60_lr0.0015_b64_fb10_softmax_labelsmooth_flip_jitter.pth",
    "osnet_x1_0_imagenet": "osnet_x1_0_imagenet.pth",
}
MARKET_URL = "https://huggingface.co/datasets/aveocr/Market-1501-v15.09.15.zip/resolve/main/Market-1501-v15.09.15.zip"
MARKET_BYTES = 152733771   # expected size, verified from the HF API before use

In [ ]:
# 3. Obtain the project code explicitly
import os, getpass, shutil
%cd /content
shutil.rmtree("/content/zed-cv", ignore_errors=True)
if SOURCE == "git":
    url = REPO_URL
    if "github.com" in url and input("Private repo? (y/N) ").strip().lower() == "y":
        tok = getpass.getpass("GitHub read-only token (not stored): ")
        url = url.replace("https://", f"https://{tok}@")
    !git clone {url} /content/zed-cv
    %cd /content/zed-cv
    !git checkout {COMMIT}
    !git rev-parse HEAD
else:
    from google.colab import files
    print("Upload zed-cv.zip (made locally with: git archive -o zed-cv.zip HEAD)")
    files.upload()                       # lands in /content/zed-cv.zip
    !mkdir -p /content/zed-cv && cd /content/zed-cv && unzip -q -o /content/zed-cv.zip
    %cd /content/zed-cv
assert os.path.exists("src/zed_cv/reid/embedder.py"), "project code not found"

In [ ]:
# 4. Pinned dependencies (torch/numpy/pillow/matplotlib come from the Colab image; versions are printed for the record)
!pip install -q scikit-learn==1.9.1 scipy==1.18.1
import sys; sys.path.insert(0, "/content/zed-cv/src")
import numpy, scipy, sklearn, PIL, matplotlib
print({m.__name__: m.__version__ for m in (numpy, scipy, sklearn, PIL, matplotlib, torch)})

In [ ]:
# 5. Weights + MOT17 labels (GT + public detections, 10 MB) + images of the SDP sequences only (HTTP range reads of MOT17.zip)
import os
os.makedirs("weights", exist_ok=True); os.makedirs("data", exist_ok=True)
tag = "osnet_x1_0_msmt17"
if not os.path.exists(f"weights/{tag}.pth"):
    !curl -sL -o weights/{tag}.pth "{HF_W}{WEIGHT_FILES[tag]}"
if not os.path.exists("data/mot17_labels/train"):
    !curl -sL -o data/mot17_labels.zip https://motchallenge.net/data/MOT17Labels.zip
    !unzip -q data/mot17_labels.zip -d data/mot17_labels
SEQS = [f"MOT17-{n}-SDP" for n in ("02", "04", "05", "09", "10", "11", "13")]
!python scripts/fetch_mot17_seq.py --seqs {" ".join(SEQS)} --out data/mot17_img

In [ ]:
# 6. GPU embedding of all public-detection crops (original + flipped view)
!python scripts/extract_mot_embeddings.py --seqs {" ".join(SEQS)} --img-root data/mot17_img --label-root data/mot17_labels/train \
    --arch x1_0 --weights weights/{tag}.pth --out artifacts/mot_embeddings/x1_0_msmt17 --batch 256

In [ ]:
# 7. Package + download (put the contents under artifacts/mot_embeddings/ in the local repo)
!zip -qr ps2_mot_embeddings.zip artifacts/mot_embeddings
from google.colab import files
files.download("ps2_mot_embeddings.zip")